# Control Variables - 16a-h Missionary stations
10/07/2026, Kuba Kowalski 

### 16a-k Missionary stations: Pre-processing cell

Combines Catholic and Protestant missionary stations into one dataset. 

In [31]:
# 16 Missions - preprocessing cell
# Combines Catholic and Protestant mission station datasets into one cleaned GeoJSON + CSV

import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

catholic_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\16_missions\Catholic Missions in Colonial Africa\AH1929_stations.csv"
)

protestant_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\16_missions\replication-data-for-the-empire-within\WMA1925_stations.xlsx"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\16_missions"
)
out_dir.mkdir(parents=True, exist_ok=True)

output_geojson = out_dir / "16_missions_combined_clean.geojson"
output_csv = out_dir / "16_missions_combined_clean.csv"

# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

catholic = pd.read_csv(catholic_file)
protestant = pd.read_excel(protestant_file)

# ------------------------------------------------------------------
# STANDARDIZE CATHOLIC DATA
# ------------------------------------------------------------------

catholic = catholic.copy()

catholic["denomination"] = "Catholic"
catholic["source_dataset"] = "AH1929"
catholic["enddate"] = np.nan
catholic["active"] = np.nan

# Staff data unavailable for Catholic missions
catholic["men"] = np.nan
catholic["wives"] = np.nan
catholic["single_women"] = np.nan
catholic["doctors_men"] = np.nan
catholic["doctors_women"] = np.nan
catholic["mission_staff_count"] = np.nan

# ------------------------------------------------------------------
# STANDARDIZE PROTESTANT DATA
# ------------------------------------------------------------------

protestant = protestant.copy()

protestant["denomination"] = "Protestant"
protestant["source_dataset"] = "WMA1925"

staff_cols = [
    "men",
    "wives",
    "single_women",
    "doctors_men",
    "doctors_women",
]

for col in staff_cols:
    protestant[col] = pd.to_numeric(protestant[col], errors="coerce").fillna(0)

protestant["mission_staff_count"] = protestant[staff_cols].sum(axis=1)

# ------------------------------------------------------------------
# KEEP COMMON COLUMNS
# ------------------------------------------------------------------

common_cols = [
    "id",
    "denomination",
    "source_dataset",
    "startdate",
    "enddate",
    "active",
    "origin",
    "colony",
    "men",
    "wives",
    "single_women",
    "doctors_men",
    "doctors_women",
    "mission_staff_count",
    "longitude",
    "latitude",
]

catholic = catholic[common_cols].copy()
protestant = protestant[common_cols].copy()

missions = pd.concat(
    [catholic, protestant],
    ignore_index=True
)

# ------------------------------------------------------------------
# CLEAN TYPES AND COORDINATES
# ------------------------------------------------------------------

missions["startdate"] = pd.to_numeric(missions["startdate"], errors="coerce")
missions["enddate"] = pd.to_numeric(missions["enddate"], errors="coerce")
missions["longitude"] = pd.to_numeric(missions["longitude"], errors="coerce")
missions["latitude"] = pd.to_numeric(missions["latitude"], errors="coerce")

missions = missions[
    missions["longitude"].notna() &
    missions["latitude"].notna()
].copy()

missions = missions[
    missions["longitude"].between(-180, 180) &
    missions["latitude"].between(-90, 90)
].copy()

# Optional readable ID to avoid Catholic/Protestant ID collisions
missions["mission_uid"] = (
    missions["denomination"].str.lower()
    + "_"
    + missions["id"].astype(str)
)

# ------------------------------------------------------------------
# CREATE GEODATAFRAME
# ------------------------------------------------------------------

missions_gdf = gpd.GeoDataFrame(
    missions,
    geometry=gpd.points_from_xy(
        missions["longitude"],
        missions["latitude"]
    ),
    crs="EPSG:4326"
)

missions_gdf = missions_gdf[
    missions_gdf.geometry.notna() &
    ~missions_gdf.geometry.is_empty
].copy()

# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

missions_gdf.to_file(output_geojson, driver="GeoJSON")
missions_gdf.drop(columns="geometry").to_csv(output_csv, index=False)

print(f"Saved GeoJSON: {output_geojson}")
print(f"Saved CSV: {output_csv}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nTotal missions:", len(missions_gdf))

print("\nMissions by denomination:")
print(missions_gdf["denomination"].value_counts(dropna=False))

print("\nStaff count availability:")
print(
    missions_gdf
    .assign(has_staff=missions_gdf["mission_staff_count"].notna())
    .groupby("denomination")["has_staff"]
    .value_counts()
)

print("\nCoordinate bounds:")
print(missions_gdf.total_bounds)

print("\nPreview:")
print(missions_gdf.head())

Saved GeoJSON: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\16_missions\16_missions_combined_clean.geojson
Saved CSV: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\16_missions\16_missions_combined_clean.csv

Total missions: 2665

Missions by denomination:
denomination
Protestant    1895
Catholic       770
Name: count, dtype: int64

Staff count availability:
denomination  has_staff
Catholic      False         770
Protestant    True         1895
Name: count, dtype: int64

Coordinate bounds:
[-27.20616541 -34.61537757  57.66983763  38.66702997]

Preview:
   id denomination source_dataset  startdate  enddate  active origin   colony  \
0   1     Catholic         AH1929     1908.0      NaN     NaN  Italy  Morocco   
1   2     Catholic         AH1929     1908.0      NaN     NaN  Italy  Morocco   
2   3     Catholic         AH1929        NaN      NaN     NaN    NaN  Morocco   
3   4     Catholic         

### 16 Missions - Group 1
- 16a mission dummy
- 16b distance to nearest mission, km

In [32]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import BallTree
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

# Reuse output from preprocessing cell
missions_file = out_dir / "16_missions_combined_clean.geojson"

group1_out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions"
)
group1_out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"

dummy_var = "16a_mission-dummy"
distance_var = "16b_distance-to-nearest-mission-km"
nearest_name_var = "16b_nearest-mission-id"

In [33]:
# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

provinces = gpd.read_file(province_polygons)
missions = gpd.read_file(missions_file)

if provinces.crs is None:
    raise ValueError("Province file has no CRS.")

if missions.crs is None:
    raise ValueError("Mission file has no CRS.")

if zone_id not in provinces.columns:
    raise ValueError(f"Column '{zone_id}' not found in province file.")

provinces[zone_id] = provinces[zone_id].astype(str).str.strip().str.zfill(6)

# Clean geometries
provinces["geometry"] = provinces.geometry.make_valid()
missions["geometry"] = missions.geometry.make_valid()

provinces = provinces[
    provinces.geometry.notna() &
    ~provinces.geometry.is_empty
].copy()

missions = missions[
    missions.geometry.notna() &
    ~missions.geometry.is_empty
].copy()

# For province-level controls, compute once per GEOLEVEL1
provinces_unique = provinces.dissolve(
    by=zone_id,
    as_index=False
)

# Match CRS for spatial join
if missions.crs != provinces_unique.crs:
    missions_join = missions.to_crs(provinces_unique.crs)
else:
    missions_join = missions.copy()

print("Province-cohort rows:", len(provinces))
print("Unique provinces:", len(provinces_unique))
print("Mission points:", len(missions_join))

Province-cohort rows: 2118
Unique provinces: 304
Mission points: 2665


In [34]:
# ------------------------------------------------------------------
# 16a MISSION DUMMY
# ------------------------------------------------------------------

intersections = gpd.sjoin(
    provinces_unique[[zone_id, "geometry"]],
    missions_join[["geometry"]],
    how="left",
    predicate="intersects"
)

mission_presence = (
    intersections
    .groupby(zone_id)["index_right"]
    .apply(lambda x: int(x.notna().any()))
    .reset_index(name=dummy_var)
)

# ------------------------------------------------------------------
# 16b DISTANCE TO NEAREST MISSION
# ------------------------------------------------------------------

# Use equal-area projection for centroids, then WGS84 for haversine distance
africa_equal_area = (
    "+proj=aea +lat_1=-18 +lat_2=21 +lat_0=0 +lon_0=20 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

provinces_projected = provinces_unique.to_crs(africa_equal_area)
centroids_projected = provinces_projected.geometry.centroid

centroids_gdf = gpd.GeoDataFrame(
    provinces_unique[[zone_id]].copy(),
    geometry=centroids_projected,
    crs=africa_equal_area
).to_crs("EPSG:4326")

centroids_gdf["centroid_latitude"] = centroids_gdf.geometry.y
centroids_gdf["centroid_longitude"] = centroids_gdf.geometry.x

# Mission coordinates in WGS84
missions_wgs84 = missions.to_crs("EPSG:4326").copy()

missions_wgs84["mission_latitude"] = missions_wgs84.geometry.y
missions_wgs84["mission_longitude"] = missions_wgs84.geometry.x

missions_wgs84 = missions_wgs84[
    missions_wgs84[["mission_latitude", "mission_longitude"]]
    .notna()
    .all(axis=1)
].copy()

centroids_gdf = centroids_gdf[
    centroids_gdf[["centroid_latitude", "centroid_longitude"]]
    .notna()
    .all(axis=1)
].copy()

earth_radius_km = 6371.0088

mission_coords_rad = np.radians(
    missions_wgs84[["mission_latitude", "mission_longitude"]].to_numpy()
)

province_coords_rad = np.radians(
    centroids_gdf[["centroid_latitude", "centroid_longitude"]].to_numpy()
)

tree = BallTree(mission_coords_rad, metric="haversine")

dist_rad, nearest_idx = tree.query(province_coords_rad, k=1)

dist_km = dist_rad.flatten() * earth_radius_km
nearest_idx = nearest_idx.flatten()

nearest_missions = missions_wgs84.iloc[nearest_idx].reset_index(drop=True)

mission_distance = pd.DataFrame({
    zone_id: centroids_gdf[zone_id].values,
    distance_var: dist_km,
    nearest_name_var: nearest_missions["mission_uid"].values
})

# ------------------------------------------------------------------
# COMBINE GROUP 1 OUTPUTS
# ------------------------------------------------------------------

group1_df = (
    mission_presence
    .merge(mission_distance, on=zone_id, how="outer")
)

group1_df[dummy_var] = group1_df[dummy_var].fillna(0).astype(int)

# ------------------------------------------------------------------
# JOIN BACK TO SPATIAL FILE FOR INSPECTION
# ------------------------------------------------------------------

provinces_mission_group1 = provinces.merge(
    group1_df,
    on=zone_id,
    how="left"
)


In [35]:
# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

csv_path = group1_out_dir / "16_group1_mission_dummy_distance.csv"
gpkg_path = group1_out_dir / "16_group1_mission_dummy_distance.gpkg"

group1_df.to_csv(csv_path, index=False)
provinces_mission_group1.to_file(gpkg_path, driver="GPKG")

print(f"\nCSV saved to: {csv_path}")
print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nMission dummy counts:")
print(group1_df[dummy_var].value_counts(dropna=False))

print("\nDistance summary:")
print(group1_df[distance_var].describe())

print("\nPreview:")
print(group1_df.head())

print("Done.")


CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group1_mission_dummy_distance.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group1_mission_dummy_distance.gpkg

Mission dummy counts:
16a_mission-dummy
1    194
0    110
Name: count, dtype: int64

Distance summary:
count    304.000000
mean      74.354439
std       92.535065
min        0.655492
25%       18.047639
50%       43.812849
75%       98.792996
max      732.552326
Name: 16b_distance-to-nearest-mission-km, dtype: float64

Preview:
  GEOLEVEL1  16a_mission-dummy  16b_distance-to-nearest-mission-km  \
0    000001                  0                           28.709555   
1    000002                  1                            6.559478   
2    000003                  1                            2.707393   
3    000004                  1                           16.450278   
4 

In [36]:
# vis for group 1

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

africa_outline_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"
)

map_output_dir = group1_out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------

group1_gpkg = group1_out_dir / "16_group1_mission_dummy_distance.gpkg"

if "provinces_mission_group1" not in globals():
    provinces_mission_group1 = gpd.read_file(group1_gpkg)

africa = gpd.read_file(africa_outline_file)

if africa.crs != provinces_mission_group1.crs:
    africa = africa.to_crs(provinces_mission_group1.crs)

# Avoid duplicate cohort geometries in visualization
provinces_plot_base = (
    provinces_mission_group1
    .sort_values(zone_id)
    .drop_duplicates(subset=[zone_id])
    .copy()
)

# ------------------------------------------------------------------
# 16a DUMMY MAP
# ------------------------------------------------------------------

var = "16a_mission-dummy"

fig, ax = plt.subplots(figsize=(16, 20))

africa.plot(
    ax=ax,
    facecolor="none",
    edgecolor="lightgrey",
    linewidth=0.5
)

provinces_plot_base.plot(
    column=var,
    cmap="viridis",
    linewidth=0.4,
    edgecolor="black",
    legend=True,
    categorical=True,
    ax=ax
)

ax.set_title("Mission presence dummy by province")
ax.set_axis_off()

minx, miny, maxx, maxy = africa.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

out_file = map_output_dir / "16a_mission_dummy_map.png"

plt.savefig(
    out_file,
    dpi=800,
    bbox_inches="tight"
)

plt.close()

print(f"Saved: {out_file}")

# ------------------------------------------------------------------
# 16b DISTANCE MAP
# ------------------------------------------------------------------

var = "16b_distance-to-nearest-mission-km"
label = "Distance to nearest mission station"

gdf_plot = provinces_plot_base[provinces_plot_base[var].notna()].copy()
values = gdf_plot[var].dropna().values

if len(values) == 0:
    print(f"Skipping {var}: no valid values")

else:
    vmin = values.min()
    vmax = values.max()

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(label)
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )

    cbar.set_label("Kilometers")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5,
        0.02,
        f"Mean: {mean_val:.3f} km   Min: {min_val:.3f} km   Max: {max_val:.3f} km",
        ha="center",
        fontsize=10
    )

    if len(values) > 1 and vmin < vmax:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = map_output_dir / "16b_distance_to_nearest_mission_map.png"

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

print("Done.")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16a_mission_dummy_map.png
Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16b_distance_to_nearest_mission_map.png
Done.


### 16 Missions - Group 2
- 16c mission staff count
- 16d mission staff density

In [37]:
# ------------------------------------------------------------------
# PATHS & LOAD DATA
# ------------------------------------------------------------------

# Run previous cells. 

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

staff_count_var = "16c_mission-staff-count"
staff_density_var = "16d_mission-staff-density"

# ------------------------------------------------------------------
# PREPARE PROVINCES AND MISSIONS
# ------------------------------------------------------------------

# Reuse provinces and missions from Group 1 if available
provinces_staff = provinces.copy()
missions_staff = missions.copy()

provinces_staff[zone_id] = (
    provinces_staff[zone_id]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)

provinces_staff["geometry"] = provinces_staff.geometry.make_valid()
missions_staff["geometry"] = missions_staff.geometry.make_valid()

provinces_staff = provinces_staff[
    provinces_staff.geometry.notna() &
    ~provinces_staff.geometry.is_empty
].copy()

missions_staff = missions_staff[
    missions_staff.geometry.notna() &
    ~missions_staff.geometry.is_empty
].copy()

# One geometry per province
provinces_unique_staff = provinces_staff.dissolve(
    by=zone_id,
    as_index=False
)

# Match CRS
if missions_staff.crs != provinces_unique_staff.crs:
    missions_staff = missions_staff.to_crs(provinces_unique_staff.crs)

# ------------------------------------------------------------------
# AREA IN KM²
# ------------------------------------------------------------------

area_crs = "EPSG:6933"

provinces_area = provinces_unique_staff.to_crs(area_crs).copy()
provinces_area["province_area_km2"] = provinces_area.geometry.area / 1_000_000

province_area_df = provinces_area[
    [zone_id, "province_area_km2"]
].copy()

# ------------------------------------------------------------------
# STAFF DATA
# ------------------------------------------------------------------

# Staff data exist only for Protestant missions.
# Catholic missions have mission_staff_count = NaN and are excluded here.
missions_staff["mission_staff_count"] = pd.to_numeric(
    missions_staff["mission_staff_count"],
    errors="coerce"
)

missions_with_staff = missions_staff[
    missions_staff["mission_staff_count"].notna()
].copy()

print("Missions with observed staff:", len(missions_with_staff))

# ------------------------------------------------------------------
# SPATIAL JOIN: STAFFED MISSIONS WITHIN PROVINCES
# ------------------------------------------------------------------

staff_join = gpd.sjoin(
    missions_with_staff[["mission_staff_count", "geometry"]],
    provinces_unique_staff[[zone_id, "geometry"]],
    how="left",
    predicate="within"
)

staff_by_province = (
    staff_join
    .dropna(subset=[zone_id])
    .groupby(zone_id)["mission_staff_count"]
    .sum()
    .reset_index(name=staff_count_var)
)

# ------------------------------------------------------------------
# COMPLETE OUTPUT FOR ALL PROVINCES
# ------------------------------------------------------------------

group2_df = (
    provinces_unique_staff[[zone_id]]
    .merge(staff_by_province, on=zone_id, how="left")
    .merge(province_area_df, on=zone_id, how="left")
)

group2_df[staff_count_var] = (
    group2_df[staff_count_var]
    .fillna(0)
    .astype(float)
)

group2_df[staff_density_var] = (
    group2_df[staff_count_var] /
    group2_df["province_area_km2"]
)

group2_df = group2_df[
    [zone_id, staff_count_var, staff_density_var]
].copy()

# ------------------------------------------------------------------
# JOIN BACK TO SPATIAL FILE FOR INSPECTION
# ------------------------------------------------------------------

provinces_mission_group2 = provinces_staff.merge(
    group2_df,
    on=zone_id,
    how="left"
)

# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

csv_path = group1_out_dir / "16_group2_mission_staff.csv"
gpkg_path = group1_out_dir / "16_group2_mission_staff.gpkg"

group2_df.to_csv(csv_path, index=False)
provinces_mission_group2.to_file(gpkg_path, driver="GPKG")

print(f"\nCSV saved to: {csv_path}")
print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nStaff count summary:")
print(group2_df[staff_count_var].describe())

print("\nStaff density summary:")
print(group2_df[staff_density_var].describe())

print("\nProvinces with nonzero Protestant mission staff:")
print((group2_df[staff_count_var] > 0).sum())

print("\nPreview:")
print(group2_df.head())

print("Done.")

Missions with observed staff: 1895

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group2_mission_staff.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group2_mission_staff.gpkg

Staff count summary:
count    304.000000
mean       5.921053
std       12.141915
min        0.000000
25%        0.000000
50%        0.000000
75%        6.000000
max      101.000000
Name: 16c_mission-staff-count, dtype: float64

Staff density summary:
count    304.000000
mean       0.000977
std        0.003682
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000391
max        0.037063
Name: 16d_mission-staff-density, dtype: float64

Provinces with nonzero Protestant mission staff:
131

Preview:
  GEOLEVEL1  16c_mission-staff-count  16d_mission-staff-density
0    000001                      0.0                   0.000000
1    000002   

In [38]:
# ------------------------------------------------------------------
# PREPARE PROVINCES AND MISSIONS
# ------------------------------------------------------------------

# Reuse provinces and missions from Group 1 if available
provinces_staff = provinces.copy()
missions_staff = missions.copy()

provinces_staff[zone_id] = (
    provinces_staff[zone_id]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)

provinces_staff["geometry"] = provinces_staff.geometry.make_valid()
missions_staff["geometry"] = missions_staff.geometry.make_valid()

provinces_staff = provinces_staff[
    provinces_staff.geometry.notna() &
    ~provinces_staff.geometry.is_empty
].copy()

missions_staff = missions_staff[
    missions_staff.geometry.notna() &
    ~missions_staff.geometry.is_empty
].copy()

# One geometry per province
provinces_unique_staff = provinces_staff.dissolve(
    by=zone_id,
    as_index=False
)

# Match CRS
if missions_staff.crs != provinces_unique_staff.crs:
    missions_staff = missions_staff.to_crs(provinces_unique_staff.crs)

# ------------------------------------------------------------------
# AREA IN KM²
# ------------------------------------------------------------------

area_crs = "EPSG:6933"

provinces_area = provinces_unique_staff.to_crs(area_crs).copy()
provinces_area["province_area_km2"] = provinces_area.geometry.area / 1_000_000

province_area_df = provinces_area[
    [zone_id, "province_area_km2"]
].copy()

# ------------------------------------------------------------------
# STAFF DATA
# ------------------------------------------------------------------

# Staff data exist only for Protestant missions.
# Catholic missions have mission_staff_count = NaN and are excluded here.
missions_staff["mission_staff_count"] = pd.to_numeric(
    missions_staff["mission_staff_count"],
    errors="coerce"
)

missions_with_staff = missions_staff[
    missions_staff["mission_staff_count"].notna()
].copy()

print("Missions with observed staff:", len(missions_with_staff))

# ------------------------------------------------------------------
# SPATIAL JOIN: STAFFED MISSIONS WITHIN PROVINCES
# ------------------------------------------------------------------

staff_join = gpd.sjoin(
    missions_with_staff[["mission_staff_count", "geometry"]],
    provinces_unique_staff[[zone_id, "geometry"]],
    how="left",
    predicate="within"
)

staff_by_province = (
    staff_join
    .dropna(subset=[zone_id])
    .groupby(zone_id)["mission_staff_count"]
    .sum()
    .reset_index(name=staff_count_var)
)

# ------------------------------------------------------------------
# COMPLETE OUTPUT FOR ALL PROVINCES
# ------------------------------------------------------------------

group2_df = (
    provinces_unique_staff[[zone_id]]
    .merge(staff_by_province, on=zone_id, how="left")
    .merge(province_area_df, on=zone_id, how="left")
)

group2_df[staff_count_var] = (
    group2_df[staff_count_var]
    .fillna(0)
    .astype(float)
)

group2_df[staff_density_var] = (
    group2_df[staff_count_var] /
    group2_df["province_area_km2"]
)

group2_df = group2_df[
    [zone_id, staff_count_var, staff_density_var]
].copy()

# ------------------------------------------------------------------
# JOIN BACK TO SPATIAL FILE FOR INSPECTION
# ------------------------------------------------------------------

provinces_mission_group2 = provinces_staff.merge(
    group2_df,
    on=zone_id,
    how="left"
)


Missions with observed staff: 1895


In [39]:
# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

csv_path = group1_out_dir / "16_group2_mission_staff.csv"
gpkg_path = group1_out_dir / "16_group2_mission_staff.gpkg"

group2_df.to_csv(csv_path, index=False)
provinces_mission_group2.to_file(gpkg_path, driver="GPKG")

print(f"\nCSV saved to: {csv_path}")
print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nStaff count summary:")
print(group2_df[staff_count_var].describe())

print("\nStaff density summary:")
print(group2_df[staff_density_var].describe())

print("\nProvinces with nonzero Protestant mission staff:")
print((group2_df[staff_count_var] > 0).sum())

print("\nPreview:")
print(group2_df.head())

print("Done.")


CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group2_mission_staff.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group2_mission_staff.gpkg

Staff count summary:
count    304.000000
mean       5.921053
std       12.141915
min        0.000000
25%        0.000000
50%        0.000000
75%        6.000000
max      101.000000
Name: 16c_mission-staff-count, dtype: float64

Staff density summary:
count    304.000000
mean       0.000977
std        0.003682
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000391
max        0.037063
Name: 16d_mission-staff-density, dtype: float64

Provinces with nonzero Protestant mission staff:
131

Preview:
  GEOLEVEL1  16c_mission-staff-count  16d_mission-staff-density
0    000001                      0.0                   0.000000
1    000002                     10.0             

In [40]:
# vis

# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------

group2_gpkg = group1_out_dir / "16_group2_mission_staff.gpkg"

if "provinces_mission_group2" not in globals():
    provinces_mission_group2 = gpd.read_file(group2_gpkg)

if "africa" not in globals():
    africa = gpd.read_file(africa_outline_file)

if africa.crs != provinces_mission_group2.crs:
    africa = africa.to_crs(provinces_mission_group2.crs)

map_output_dir = group1_out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# Avoid duplicate cohort geometries in visualization
provinces_plot_base = (
    provinces_mission_group2
    .sort_values(zone_id)
    .drop_duplicates(subset=[zone_id])
    .copy()
)

# ------------------------------------------------------------------
# VARIABLES TO MAP
# ------------------------------------------------------------------

variables = {
    "16c_mission-staff-count": {
        "title": "Protestant mission staff count by province",
        "cbar": "Mission staff count"
    },
    "16d_mission-staff-density": {
        "title": "Protestant mission staff density by province",
        "cbar": "Mission staff per km²"
    }
}

for var, meta in variables.items():

    gdf_plot = provinces_plot_base[provinces_plot_base[var].notna()].copy()
    values = gdf_plot[var].dropna().values

    if len(values) == 0:
        print(f"Skipping {var}: no valid values")
        continue

    vmin = values.min()
    vmax = values.max()

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(meta["title"])
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )

    cbar.set_label(meta["cbar"])

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5,
        0.02,
        f"Mean: {mean_val:.3f}   Min: {min_val:.3f}   Max: {max_val:.3f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1 and vmin < vmax:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    else:
        constant_val = values[0] if len(values) > 0 else np.nan

        if np.isfinite(constant_val):
            wave_ax.axvline(
                constant_val,
                linewidth=1.5,
                linestyle="--"
            )

            wave_ax.text(
                constant_val,
                0.5,
                "constant",
                ha="center",
                va="center",
                fontsize=8,
                transform=wave_ax.get_xaxis_transform()
            )

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    safe_var_name = var.replace("-", "_")
    out_file = map_output_dir / f"{safe_var_name}_map.png"

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

print("Done.")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16c_mission_staff_count_map.png
Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16d_mission_staff_density_map.png
Done.


### 16 Missions - Group 3
- 16e 25 km exposure buffer (Share of province area that falls within a 25 km buffer of any mission station)

In [41]:
# ------------------------------------------------------------------
# PATHS & LOAD DATA
# ------------------------------------------------------------------

# Run previous cells. 

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

exposure_var = "16e_25-km-mission-exposure-buffer"

buffer_crs = "EPSG:6933"   # meters
buffer_distance_m = 25_000 # Change this value if you want a different buffer distance (in meters)
# ------------------------------------------------------------------
# PREPARE DATA
# ------------------------------------------------------------------

provinces_buffer = provinces.copy()
missions_buffer = missions.copy()

provinces_buffer[zone_id] = (
    provinces_buffer[zone_id]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)

provinces_buffer["geometry"] = provinces_buffer.geometry.make_valid()
missions_buffer["geometry"] = missions_buffer.geometry.make_valid()

provinces_buffer = provinces_buffer[
    provinces_buffer.geometry.notna() &
    ~provinces_buffer.geometry.is_empty
].copy()

missions_buffer = missions_buffer[
    missions_buffer.geometry.notna() &
    ~missions_buffer.geometry.is_empty
].copy()

# One geometry per province
provinces_unique_buffer = provinces_buffer.dissolve(
    by=zone_id,
    as_index=False
)

# Project to meter-based CRS
provinces_m = provinces_unique_buffer.to_crs(buffer_crs)
missions_m = missions_buffer.to_crs(buffer_crs)

# ------------------------------------------------------------------
# CREATE DISSOLVED 25 KM MISSION BUFFER
# ------------------------------------------------------------------

missions_m["geometry"] = missions_m.geometry.buffer(buffer_distance_m)

mission_buffer_union = gpd.GeoDataFrame(
    {"buffer_id": [1]},
    geometry=[missions_m.geometry.union_all()],
    crs=buffer_crs
)

# ------------------------------------------------------------------
# CALCULATE PROVINCE AREA
# ------------------------------------------------------------------

provinces_m["province_area_m2"] = provinces_m.geometry.area

# ------------------------------------------------------------------
# INTERSECT PROVINCES WITH MISSION BUFFER
# ------------------------------------------------------------------

intersection = gpd.overlay(
    provinces_m[[zone_id, "province_area_m2", "geometry"]],
    mission_buffer_union[["buffer_id", "geometry"]],
    how="intersection",
    keep_geom_type=True
)

if len(intersection) > 0:
    intersection["buffer_area_m2"] = intersection.geometry.area

    exposure_by_province = (
        intersection
        .groupby(zone_id)["buffer_area_m2"]
        .sum()
        .reset_index()
    )

else:
    exposure_by_province = pd.DataFrame(
        columns=[zone_id, "buffer_area_m2"]
    )

# ------------------------------------------------------------------
# COMPLETE OUTPUT FOR ALL PROVINCES
# ------------------------------------------------------------------

group3_df = (
    provinces_m[[zone_id, "province_area_m2"]]
    .merge(exposure_by_province, on=zone_id, how="left")
)

group3_df["buffer_area_m2"] = group3_df["buffer_area_m2"].fillna(0)

group3_df[exposure_var] = (
    group3_df["buffer_area_m2"] /
    group3_df["province_area_m2"]
)

group3_df[exposure_var] = group3_df[exposure_var].clip(0, 1)

group3_df = group3_df[
    [zone_id, exposure_var]
].copy()

# ------------------------------------------------------------------
# JOIN BACK TO SPATIAL FILE FOR INSPECTION
# ------------------------------------------------------------------

provinces_mission_group3 = provinces_buffer.merge(
    group3_df,
    on=zone_id,
    how="left"
)

# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

csv_path = group1_out_dir / "16_group3_mission_25km_exposure_buffer.csv"
gpkg_path = group1_out_dir / "16_group3_mission_25km_exposure_buffer.gpkg"

group3_df.to_csv(csv_path, index=False)
provinces_mission_group3.to_file(gpkg_path, driver="GPKG")

print(f"\nCSV saved to: {csv_path}")
print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nExposure buffer summary:")
print(group3_df[exposure_var].describe())

print("\nProvinces with zero exposure:")
print((group3_df[exposure_var] == 0).sum())

print("\nProvinces with full exposure:")
print((group3_df[exposure_var] == 1).sum())

print("\nPreview:")
print(group3_df.head())

print("Done.")


CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group3_mission_25km_exposure_buffer.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group3_mission_25km_exposure_buffer.gpkg

Exposure buffer summary:
count    304.000000
mean       0.288788
std        0.319671
min        0.000000
25%        0.017681
50%        0.150679
75%        0.462344
max        1.000000
Name: 16e_25-km-mission-exposure-buffer, dtype: float64

Provinces with zero exposure:
49

Provinces with full exposure:
6

Preview:
  GEOLEVEL1  16e_25-km-mission-exposure-buffer
0    000001                           0.896624
1    000002                           0.849748
2    000003                           0.926955
3    000004                           0.783167
4    000005                           0.374821
Done.


In [42]:
# ------------------------------------------------------------------
# PREPARE DATA
# ------------------------------------------------------------------

provinces_buffer = provinces.copy()
missions_buffer = missions.copy()

provinces_buffer[zone_id] = (
    provinces_buffer[zone_id]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)

provinces_buffer["geometry"] = provinces_buffer.geometry.make_valid()
missions_buffer["geometry"] = missions_buffer.geometry.make_valid()

provinces_buffer = provinces_buffer[
    provinces_buffer.geometry.notna() &
    ~provinces_buffer.geometry.is_empty
].copy()

missions_buffer = missions_buffer[
    missions_buffer.geometry.notna() &
    ~missions_buffer.geometry.is_empty
].copy()

# One geometry per province
provinces_unique_buffer = provinces_buffer.dissolve(
    by=zone_id,
    as_index=False
)

# Project to meter-based CRS
provinces_m = provinces_unique_buffer.to_crs(buffer_crs)
missions_m = missions_buffer.to_crs(buffer_crs)

# ------------------------------------------------------------------
# CREATE DISSOLVED 25 KM MISSION BUFFER
# ------------------------------------------------------------------

missions_m["geometry"] = missions_m.geometry.buffer(buffer_distance_m)

mission_buffer_union = gpd.GeoDataFrame(
    {"buffer_id": [1]},
    geometry=[missions_m.geometry.union_all()],
    crs=buffer_crs
)

# ------------------------------------------------------------------
# CALCULATE PROVINCE AREA
# ------------------------------------------------------------------

provinces_m["province_area_m2"] = provinces_m.geometry.area

# ------------------------------------------------------------------
# INTERSECT PROVINCES WITH MISSION BUFFER
# ------------------------------------------------------------------

intersection = gpd.overlay(
    provinces_m[[zone_id, "province_area_m2", "geometry"]],
    mission_buffer_union[["buffer_id", "geometry"]],
    how="intersection",
    keep_geom_type=True
)

if len(intersection) > 0:
    intersection["buffer_area_m2"] = intersection.geometry.area

    exposure_by_province = (
        intersection
        .groupby(zone_id)["buffer_area_m2"]
        .sum()
        .reset_index()
    )

else:
    exposure_by_province = pd.DataFrame(
        columns=[zone_id, "buffer_area_m2"]
    )

# ------------------------------------------------------------------
# COMPLETE OUTPUT FOR ALL PROVINCES
# ------------------------------------------------------------------

group3_df = (
    provinces_m[[zone_id, "province_area_m2"]]
    .merge(exposure_by_province, on=zone_id, how="left")
)

group3_df["buffer_area_m2"] = group3_df["buffer_area_m2"].fillna(0)

group3_df[exposure_var] = (
    group3_df["buffer_area_m2"] /
    group3_df["province_area_m2"]
)

group3_df[exposure_var] = group3_df[exposure_var].clip(0, 1)

group3_df = group3_df[
    [zone_id, exposure_var]
].copy()

# ------------------------------------------------------------------
# JOIN BACK TO SPATIAL FILE FOR INSPECTION
# ------------------------------------------------------------------

provinces_mission_group3 = provinces_buffer.merge(
    group3_df,
    on=zone_id,
    how="left"
)


In [43]:
# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

csv_path = group1_out_dir / "16_group3_mission_25km_exposure_buffer.csv"
gpkg_path = group1_out_dir / "16_group3_mission_25km_exposure_buffer.gpkg"

group3_df.to_csv(csv_path, index=False)
provinces_mission_group3.to_file(gpkg_path, driver="GPKG")

print(f"\nCSV saved to: {csv_path}")
print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nExposure buffer summary:")
print(group3_df[exposure_var].describe())

print("\nProvinces with zero exposure:")
print((group3_df[exposure_var] == 0).sum())

print("\nProvinces with full exposure:")
print((group3_df[exposure_var] == 1).sum())

print("\nPreview:")
print(group3_df.head())

print("Done.")


CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group3_mission_25km_exposure_buffer.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group3_mission_25km_exposure_buffer.gpkg

Exposure buffer summary:
count    304.000000
mean       0.288788
std        0.319671
min        0.000000
25%        0.017681
50%        0.150679
75%        0.462344
max        1.000000
Name: 16e_25-km-mission-exposure-buffer, dtype: float64

Provinces with zero exposure:
49

Provinces with full exposure:
6

Preview:
  GEOLEVEL1  16e_25-km-mission-exposure-buffer
0    000001                           0.896624
1    000002                           0.849748
2    000003                           0.926955
3    000004                           0.783167
4    000005                           0.374821
Done.


In [44]:
#vis

# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------

group3_gpkg = group1_out_dir / "16_group3_mission_25km_exposure_buffer.gpkg"

if "provinces_mission_group3" not in globals():
    provinces_mission_group3 = gpd.read_file(group3_gpkg)

if "africa" not in globals():
    africa = gpd.read_file(africa_outline_file)

if africa.crs != provinces_mission_group3.crs:
    africa = africa.to_crs(provinces_mission_group3.crs)

map_output_dir = group1_out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# Avoid duplicate cohort geometries in visualization
provinces_plot_base = (
    provinces_mission_group3
    .sort_values(zone_id)
    .drop_duplicates(subset=[zone_id])
    .copy()
)

# ------------------------------------------------------------------
# MAP
# ------------------------------------------------------------------

var = "16e_25-km-mission-exposure-buffer"
label = "Share of province area within 25 km of a mission station"

gdf_plot = provinces_plot_base[provinces_plot_base[var].notna()].copy()
values = gdf_plot[var].dropna().values

if len(values) == 0:
    print(f"Skipping {var}: no valid values")

else:
    vmin = 0
    vmax = 1

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(label)
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )

    cbar.set_label("Share of province area")

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5,
        0.02,
        f"Mean: {mean_val:.3f}   Min: {min_val:.3f}   Max: {max_val:.3f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1 and values.min() < values.max():
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    out_file = map_output_dir / "16e_25_km_mission_exposure_buffer_map.png"

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

print("Done.")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16e_25_km_mission_exposure_buffer_map.png
Done.


### 16 Missions - Group 4
- 16f mission count within 25 km of province centroid
- 16g mission count within 50 km of province centroid
- 16h mission count within 75 km of province centroid
- 16i mission count within 100 km of province centroid
- 16j total mission count within province

In [45]:
# ------------------------------------------------------------------
# PATHS & LOAD DATA
# ------------------------------------------------------------------

# Run previous cells. 

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

count_25_var = "16f_mission-count-within-25km-centroid"
count_50_var = "16g_mission-count-within-50km-centroid"
count_75_var = "16h_mission-count-within-75km-centroid"
count_100_var = "16i_mission-count-within-100km-centroid"
count_total_var = "16j_mission-count-total"

# ------------------------------------------------------------------
# PREPARE DATA
# ------------------------------------------------------------------

provinces_count = provinces.copy()
missions_count = missions.copy()

provinces_count[zone_id] = (
    provinces_count[zone_id]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)

provinces_count["geometry"] = provinces_count.geometry.make_valid()
missions_count["geometry"] = missions_count.geometry.make_valid()

provinces_count = provinces_count[
    provinces_count.geometry.notna() &
    ~provinces_count.geometry.is_empty
].copy()

missions_count = missions_count[
    missions_count.geometry.notna() &
    ~missions_count.geometry.is_empty
].copy()

# One geometry per province
provinces_unique_count = provinces_count.dissolve(
    by=zone_id,
    as_index=False
)

In [46]:
# ------------------------------------------------------------------
# CENTROID-BASED MISSION COUNTS WITH BALLTREE
# ------------------------------------------------------------------

africa_equal_area = (
    "+proj=aea +lat_1=-18 +lat_2=21 +lat_0=0 +lon_0=20 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

provinces_projected = provinces_unique_count.to_crs(africa_equal_area)
centroids_projected = provinces_projected.geometry.centroid

centroids_gdf = gpd.GeoDataFrame(
    provinces_unique_count[[zone_id]].copy(),
    geometry=centroids_projected,
    crs=africa_equal_area
).to_crs("EPSG:4326")

centroids_gdf["centroid_latitude"] = centroids_gdf.geometry.y
centroids_gdf["centroid_longitude"] = centroids_gdf.geometry.x

missions_wgs84 = missions_count.to_crs("EPSG:4326").copy()
missions_wgs84["mission_latitude"] = missions_wgs84.geometry.y
missions_wgs84["mission_longitude"] = missions_wgs84.geometry.x

missions_wgs84 = missions_wgs84[
    missions_wgs84[["mission_latitude", "mission_longitude"]]
    .notna()
    .all(axis=1)
].copy()

centroids_gdf = centroids_gdf[
    centroids_gdf[["centroid_latitude", "centroid_longitude"]]
    .notna()
    .all(axis=1)
].copy()

earth_radius_km = 6371.0088

mission_coords_rad = np.radians(
    missions_wgs84[["mission_latitude", "mission_longitude"]].to_numpy()
)

province_coords_rad = np.radians(
    centroids_gdf[["centroid_latitude", "centroid_longitude"]].to_numpy()
)

tree = BallTree(mission_coords_rad, metric="haversine")

radii_km = {
    count_25_var: 25,
    count_50_var: 50,
    count_75_var: 75,
    count_100_var: 100,
}

centroid_counts = centroids_gdf[[zone_id]].copy()

for var, radius_km in radii_km.items():

    radius_rad = radius_km / earth_radius_km

    matches = tree.query_radius(
        province_coords_rad,
        r=radius_rad
    )

    centroid_counts[var] = [len(x) for x in matches]

# ------------------------------------------------------------------
# TOTAL MISSION COUNT WITHIN PROVINCE
# ------------------------------------------------------------------

if missions_count.crs != provinces_unique_count.crs:
    missions_join = missions_count.to_crs(provinces_unique_count.crs)
else:
    missions_join = missions_count.copy()

mission_join = gpd.sjoin(
    missions_join[["geometry"]],
    provinces_unique_count[[zone_id, "geometry"]],
    how="left",
    predicate="within"
)

mission_total = (
    mission_join
    .dropna(subset=[zone_id])
    .groupby(zone_id)
    .size()
    .reset_index(name=count_total_var)
)

# ------------------------------------------------------------------
# COMPLETE OUTPUT FOR ALL PROVINCES
# ------------------------------------------------------------------

group4_df = (
    provinces_unique_count[[zone_id]]
    .merge(centroid_counts, on=zone_id, how="left")
    .merge(mission_total, on=zone_id, how="left")
)

for col in [
    count_25_var,
    count_50_var,
    count_75_var,
    count_100_var,
    count_total_var
]:
    group4_df[col] = group4_df[col].fillna(0).astype(int)

# ------------------------------------------------------------------
# JOIN BACK TO SPATIAL FILE FOR INSPECTION
# ------------------------------------------------------------------

provinces_mission_group4 = provinces_count.merge(
    group4_df,
    on=zone_id,
    how="left"
)

In [47]:
# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

csv_path = group1_out_dir / "16_group4_mission_counts.csv"
gpkg_path = group1_out_dir / "16_group4_mission_counts.gpkg"

group4_df.to_csv(csv_path, index=False)
provinces_mission_group4.to_file(gpkg_path, driver="GPKG")

print(f"\nCSV saved to: {csv_path}")
print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nMission count summaries:")
print(group4_df.drop(columns=[zone_id]).describe())

print("\nPreview:")
print(group4_df.head())

print("Done.")


CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group4_mission_counts.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16_group4_mission_counts.gpkg

Mission count summaries:
       16f_mission-count-within-25km-centroid  \
count                              304.000000   
mean                                 0.796053   
std                                  1.658417   
min                                  0.000000   
25%                                  0.000000   
50%                                  0.000000   
75%                                  1.000000   
max                                 11.000000   

       16g_mission-count-within-50km-centroid  \
count                              304.000000   
mean                                 2.240132   
std                                  3.279837   
min                                

In [48]:
#vis

# ------------------------------------------------------------------
# LOAD OUTPUTS IF NEEDED
# ------------------------------------------------------------------

group4_gpkg = group1_out_dir / "16_group4_mission_counts.gpkg"

if "provinces_mission_group4" not in globals():
    provinces_mission_group4 = gpd.read_file(group4_gpkg)

if "africa" not in globals():
    africa = gpd.read_file(africa_outline_file)

if africa.crs != provinces_mission_group4.crs:
    africa = africa.to_crs(provinces_mission_group4.crs)

map_output_dir = group1_out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# Avoid duplicate cohort geometries in visualization
provinces_plot_base = (
    provinces_mission_group4
    .sort_values(zone_id)
    .drop_duplicates(subset=[zone_id])
    .copy()
)

# ------------------------------------------------------------------
# VARIABLES TO MAP
# ------------------------------------------------------------------

variables = {
    "16f_mission-count-within-25km-centroid": {
        "title": "Mission count within 25 km of province centroid",
        "cbar": "Mission count"
    },
    "16g_mission-count-within-50km-centroid": {
        "title": "Mission count within 50 km of province centroid",
        "cbar": "Mission count"
    },
    "16h_mission-count-within-75km-centroid": {
        "title": "Mission count within 75 km of province centroid",
        "cbar": "Mission count"
    },
    "16i_mission-count-within-100km-centroid": {
        "title": "Mission count within 100 km of province centroid",
        "cbar": "Mission count"
    },
    "16j_mission-count-total": {
        "title": "Total mission count within province",
        "cbar": "Mission count"
    }
}

for var, meta in variables.items():

    gdf_plot = provinces_plot_base[provinces_plot_base[var].notna()].copy()
    values = gdf_plot[var].dropna().values

    if len(values) == 0:
        print(f"Skipping {var}: no valid values")
        continue

    vmin = values.min()
    vmax = values.max()

    fig, ax = plt.subplots(figsize=(16, 20))

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(meta["title"])
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )

    cbar.set_label(meta["cbar"])

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5,
        0.02,
        f"Mean: {mean_val:.3f}   Min: {min_val:.3f}   Max: {max_val:.3f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1 and vmin < vmax:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    else:
        constant_val = values[0] if len(values) > 0 else np.nan

        if np.isfinite(constant_val):
            wave_ax.axvline(
                constant_val,
                linewidth=1.5,
                linestyle="--"
            )

            wave_ax.text(
                constant_val,
                0.5,
                "constant",
                ha="center",
                va="center",
                fontsize=8,
                transform=wave_ax.get_xaxis_transform()
            )

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    safe_var_name = var.replace("-", "_")
    out_file = map_output_dir / f"{safe_var_name}_map.png"

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

print("Done.")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16f_mission_count_within_25km_centroid_map.png
Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16g_mission_count_within_50km_centroid_map.png
Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16h_mission_count_within_75km_centroid_map.png
Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16i_mission_count_within_100km_centroid_map.png
Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16j_mission_count_total_map.png
Done.


### 16 Missions - Group 5
- 16k mission density (total mission count divided by province area)

In [49]:
# ------------------------------------------------------------------
# 16k. MISSION STATION DENSITY
# Number of mission stations per km²
# ------------------------------------------------------------------

mission_density_var = "16k_mission-station-density"

In [50]:
# ------------------------------------------------------------------
# COUNT MISSIONS PER PROVINCE
# ------------------------------------------------------------------

mission_join = gpd.sjoin(
    missions_staff[["geometry"]],
    provinces_unique_staff[[zone_id, "geometry"]],
    how="left",
    predicate="within"
)

mission_count = (
    mission_join
    .dropna(subset=[zone_id])
    .groupby(zone_id)
    .size()
    .reset_index(name="mission_count")
)

# ------------------------------------------------------------------
# CALCULATE DENSITY
# ------------------------------------------------------------------

mission_density_df = (
    provinces_unique_staff[[zone_id]]
    .merge(mission_count, on=zone_id, how="left")
    .merge(province_area_df, on=zone_id, how="left")
)

mission_density_df["mission_count"] = (
    mission_density_df["mission_count"]
    .fillna(0)
    .astype(int)
)

mission_density_df[mission_density_var] = (
    mission_density_df["mission_count"] /
    mission_density_df["province_area_km2"]
)

mission_density_df = mission_density_df[
    [zone_id, mission_density_var]
].copy()

In [51]:
# ------------------------------------------------------------------
# JOIN BACK TO SPATIAL FILE
# ------------------------------------------------------------------

provinces_mission_density = provinces_staff.merge(
    mission_density_df,
    on=zone_id,
    how="left"
)

# ------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------

csv_path = group1_out_dir / "16k_mission_station_density.csv"
gpkg_path = group1_out_dir / "16k_mission_station_density.gpkg"

mission_density_df.to_csv(csv_path, index=False)
provinces_mission_density.to_file(gpkg_path, driver="GPKG")

print(f"CSV saved to: {csv_path}")
print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nMission station density summary:")
print(mission_density_df[mission_density_var].describe())

print("\nPreview:")
print(mission_density_df.head())

print("Done.")

CSV saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16k_mission_station_density.csv
GPKG saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\16k_mission_station_density.gpkg

Mission station density summary:
count    304.000000
mean       0.000542
std        0.001566
min        0.000000
25%        0.000000
50%        0.000068
75%        0.000344
max        0.014204
Name: 16k_mission-station-density, dtype: float64

Preview:
  GEOLEVEL1  16k_mission-station-density
0    000001                     0.000000
1    000002                     0.001337
2    000003                     0.001026
3    000004                     0.000921
4    000005                     0.000211
Done.


In [52]:
# ------------------------------------------------------------------
# VISUALIZATION: MISSION STATION DENSITY
# ------------------------------------------------------------------

import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

africa_outline_file = (
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa"
    r"\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"
)

map_output_dir = group1_out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

africa = gpd.read_file(africa_outline_file)

provinces_plot_base = provinces_mission_density.copy()

if africa.crs != provinces_plot_base.crs:
    africa = africa.to_crs(provinces_plot_base.crs)

# ------------------------------------------------------------------
# VARIABLE TO MAP
# ------------------------------------------------------------------

variables = {
    "16k_mission-station-density":
        "Mission station density (missions per km²)"
}

for var, label in variables.items():

    gdf_plot = provinces_plot_base[
        provinces_plot_base[var].notna()
    ].copy()

    values = gdf_plot[var].values

    if len(values) == 0:
        print(f"Skipping {var}: no valid values")
        continue

    vmin = values.min()
    vmax = values.max()

    fig, ax = plt.subplots(figsize=(16, 20))

    # --------------------------------------------------
    # AFRICA OUTLINE
    # --------------------------------------------------

    africa.plot(
        ax=ax,
        facecolor="none",
        edgecolor="lightgrey",
        linewidth=0.5
    )

    # --------------------------------------------------
    # PROVINCES
    # --------------------------------------------------

    gdf_plot.plot(
        column=var,
        cmap="viridis",
        linewidth=0.4,
        edgecolor="black",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        ax=ax
    )

    ax.set_title(label)
    ax.set_axis_off()

    minx, miny, maxx, maxy = africa.total_bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    # --------------------------------------------------
    # COLORBAR
    # --------------------------------------------------

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    sm = mpl.cm.ScalarMappable(
        norm=norm,
        cmap="viridis"
    )
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.04
    )

    cbar.set_label("Mission stations per km²")

    # --------------------------------------------------
    # KDE ABOVE COLORBAR
    # --------------------------------------------------

    cb_pos = cbar.ax.get_position()

    wave_ax = fig.add_axes([
        cb_pos.x0,
        cb_pos.y1 + 0.005,
        cb_pos.width,
        0.05
    ])

    mean_val = values.mean()
    min_val = values.min()
    max_val = values.max()

    fig.text(
        0.5,
        0.02,
        f"Mean: {mean_val:.6f}   Min: {min_val:.6f}   Max: {max_val:.6f}",
        ha="center",
        fontsize=10
    )

    if len(values) > 1 and vmax > vmin:
        kde = gaussian_kde(values)
        x = np.linspace(vmin, vmax, 300)
        y = kde(x)

        wave_ax.plot(x, y, linewidth=1.5)
        wave_ax.fill_between(x, y, alpha=0.25)

    wave_ax.set_xlim(vmin, vmax)
    wave_ax.set_xticks([])
    wave_ax.set_yticks([])

    for spine in wave_ax.spines.values():
        spine.set_visible(False)

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------

    out_file = (
        map_output_dir /
        "16k_mission_station_density_map.png"
    )

    plt.savefig(
        out_file,
        dpi=800,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {out_file}")

print("Done.")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16k_mission_station_density_map.png
Done.


In [53]:
# ------------------------------------------------------------------
# VISUALIZATION: MISSION STATION DENSITY
# Color scale capped at 0.002 for better comparison of most of the distribution
# outliers retained on map
# ------------------------------------------------------------------

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.stats import gaussian_kde

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

africa_outline_file = (
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa"
    r"\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"
)

map_output_dir = group1_out_dir / "maps"
map_output_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------

africa = gpd.read_file(africa_outline_file)
provinces_plot_base = provinces_mission_density.copy()

if africa.crs != provinces_plot_base.crs:
    africa = africa.to_crs(provinces_plot_base.crs)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

var = "16k_mission-station-density"
label = "Mission station density (missions per km²)"
vmin = 0
vmax = 0.002

# ------------------------------------------------------------------
# PREPARE VALUES
# ------------------------------------------------------------------

gdf_plot = provinces_plot_base[
    provinces_plot_base[var].notna()
].copy()

values = gdf_plot[var].dropna().values
kde_values = values[values <= vmax]

if len(values) == 0:
    raise ValueError(f"No valid values found for {var}")

# ------------------------------------------------------------------
# MAP
# ------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(16, 20))

africa.plot(
    ax=ax,
    facecolor="none",
    edgecolor="lightgrey",
    linewidth=0.5
)

gdf_plot.plot(
    column=var,
    cmap="viridis",
    linewidth=0.4,
    edgecolor="black",
    legend=False,
    vmin=vmin,
    vmax=vmax,
    ax=ax
)

ax.set_title(label)
ax.set_axis_off()

minx, miny, maxx, maxy = africa.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

# ------------------------------------------------------------------
# COLORBAR WITH CAPPED MAXIMUM
# ------------------------------------------------------------------

norm = mpl.colors.Normalize(
    vmin=vmin,
    vmax=vmax,
    clip=True
)

sm = mpl.cm.ScalarMappable(
    norm=norm,
    cmap="viridis"
)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    orientation="horizontal",
    fraction=0.05,
    pad=0.04,
    extend="max"
)

cbar.set_label("Mission stations per km²")

# ------------------------------------------------------------------
# KDE ABOVE COLORBAR WITH PEAK LABELS
# ------------------------------------------------------------------

cb_pos = cbar.ax.get_position()

wave_ax = fig.add_axes([
    cb_pos.x0,
    cb_pos.y1 + 0.005,
    cb_pos.width,
    0.05
])

wave_ax.set_xlim(vmin, vmax)
wave_ax.set_xticks([])
wave_ax.set_yticks([])

for spine in wave_ax.spines.values():
    spine.set_visible(False)

if len(kde_values) > 1 and kde_values.min() < kde_values.max():
    kde = gaussian_kde(kde_values)
    x = np.linspace(vmin, vmax, 500)
    y = kde(x)

    wave_ax.plot(x, y, linewidth=1.5)
    wave_ax.fill_between(x, y, alpha=0.25)

    # Detect local peaks
    from scipy.signal import find_peaks

    peaks, properties = find_peaks(
        y,
        prominence=y.max() * 0.03,
        distance=25
    )

    # Label only peaks after the main concentration range
    peak_x = x[peaks]
    peak_y = y[peaks]

    peak_mask = peak_x >= 0.00075
    peak_x = peak_x[peak_mask]
    peak_y = peak_y[peak_mask]

    # Avoid too many labels
    max_labels = 6
    if len(peak_x) > max_labels:
        top_idx = np.argsort(peak_y)[-max_labels:]
        peak_x = peak_x[top_idx]
        peak_y = peak_y[top_idx]

    for px, py in zip(peak_x, peak_y):
        wave_ax.axvline(
            px,
            ymax=0.85,
            linewidth=0.8,
            linestyle="--",
            alpha=0.7
        )

        wave_ax.text(
            px,
            py,
            f"{px:.6f}",
            ha="center",
            va="bottom",
            fontsize=7,
            rotation=45
        )

# ------------------------------------------------------------------
# SAVE
# ------------------------------------------------------------------

out_file = map_output_dir / "16k_mission_station_density_map_capped_0002.png"

plt.savefig(
    out_file,
    dpi=800,
    bbox_inches="tight"
)

plt.close()

print(f"Saved: {out_file}")

Saved: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\16_missions\maps\16k_mission_station_density_map_capped_0002.png
